# Human Action Recognition — Transformer Training

Replaces the LSTM notebook (`lstm_train.ipynb`) with a Spatiotemporal Transformer.

**Architecture summary**
```
Input (B, 32, 36)
  └─ Linear 36→128          patch / frame embedding
  └─ Sinusoidal PositionalEncoding
  └─ 4 × TransformerEncoderLayer (d=128, heads=8, ffn=256, drop=0.1)
  └─ Global Average Pool over T
  └─ Linear 128→64 → ReLU → Linear 64→6
```
6 classes: JUMPING, JUMPING_JACKS, BOXING, WAVING_2HANDS, WAVING_1HAND, CLAPPING_HANDS


## 1 · Environment setup

In [16]:
# Run once in Colab
!pip install pytorch-lightning einops --quiet

## 2 · Download & unzip dataset

In [17]:
import os
import sys
import zipfile
import subprocess

# Install gdown if not installed
try:
    import gdown
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "gdown",
        "--quiet"
    ])
    import gdown

# Google Drive file
url = "https://drive.google.com/uc?id=1IuZlyNjg6DMQE3iaO1Px6h1yLKgatynt"
zip_path = "RNN-HAR-2D-Pose-database.zip"
extract_folder = "RNN-HAR-2D-Pose-database"

# Download zip if not already downloaded
if not os.path.exists(zip_path):
    gdown.download(url, zip_path, quiet=False)

# Extract zip if not already extracted
if not os.path.exists(extract_folder):
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(".")

print("Dataset ready successfully.")

Dataset ready successfully.


## 3 · Clone repo & install deps

In [18]:

!pip install torch torchvision transformers opencv-python mediapipe numpy pandas scikit-learn flask

## 4 · Copy the transformer module into src/

In [19]:
# Paste the full contents of src/transformer.py here, or upload the file.
# If you already placed transformer.py in the repo's src/ folder, skip this cell.
import os, textwrap
os.makedirs('src', exist_ok=True)

TRANSFORMER_CODE = '''
# ── src/transformer.py ───────────────────────────────────────────────────────
# (copy the full src/transformer.py content here if not already present)
'''

# Preferred: just upload src/transformer.py and skip this cell.

## 5 · Config

In [20]:
from argparse import ArgumentParser

DATASET_PATH = "content/RNN-HAR-2D-Pose-database"

HPARAMS = dict(
    batch_size      = 256,
    epochs          = 50,
    learning_rate   = 1e-4,
    num_classes     = 6,
    d_model         = 128,
    nhead           = 8,
    num_layers      = 4,
    dim_feedforward = 256,
    dropout         = 0.1,
    input_size      = 36,   # 18 keypoints ? 2
)
print(HPARAMS)

{'batch_size': 256, 'epochs': 50, 'learning_rate': 0.0001, 'num_classes': 6, 'd_model': 128, 'nhead': 8, 'num_layers': 4, 'dim_feedforward': 256, 'dropout': 0.1, 'input_size': 36}


## 6 · Import model & data module

In [21]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor

# NEW import — transformer replaces lstm
from transformer import ActionClassificationTransformer, PoseDataModule

## 7 · Build model + data module

In [22]:
pl.seed_everything(42)

model = ActionClassificationTransformer(
    input_size      = HPARAMS['input_size'],
    d_model         = HPARAMS['d_model'],
    nhead           = HPARAMS['nhead'],
    num_layers      = HPARAMS['num_layers'],
    dim_feedforward = HPARAMS['dim_feedforward'],
    dropout         = HPARAMS['dropout'],
    num_classes     = HPARAMS['num_classes'],
    learning_rate   = HPARAMS['learning_rate'],
)

data_module = PoseDataModule(
    data_root  = DATASET_PATH,
    batch_size = HPARAMS['batch_size'],
)

print(model)

Seed set to 42


ActionClassificationTransformer(
  (input_proj): Linear(in_features=36, out_features=128, bias=True)
  (pos_enc): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (classifier): Linear(in_features

## 8 · Callbacks

In [23]:
checkpoint_callback = ModelCheckpoint(
    monitor   = 'val_loss',
    save_top_k= 1,
    filename  = 'transformer-{epoch:02d}-{val_loss:.4f}',
    verbose   = True,
)

early_stop = EarlyStopping(
    monitor  = 'val_loss',
    patience = 10,
    verbose  = True,
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

## 9 · TensorBoard (optional)

In [24]:
try:
    from tensorboard import default  # noqa: F401
    print("TensorBoard is available. Run `%load_ext tensorboard` and `%tensorboard --logdir=lightning_logs` in a notebook cell if you want the dashboard.")
except Exception as exc:
    print(f"TensorBoard dashboard skipped: {exc}")

TensorBoard dashboard skipped: No module named 'pkg_resources'


In [25]:
!pip install setuptools
!pip install --upgrade tensorboard tensorboard-data-server

## 10 · Train

In [26]:
import torch

trainer = pl.Trainer(
    max_epochs         = HPARAMS['epochs'],
    accelerator        = "gpu" if torch.cuda.is_available() else "cpu",
    devices            = 1,
    deterministic      = True,
    enable_progress_bar= True,
    callbacks          = [
        early_stop,
        checkpoint_callback,
        lr_monitor
    ],
)

trainer.fit(model, data_module)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ input_proj          │ Linear             │  4.7 K │ train │     0 │
│ 1 │ pos_enc             │ PositionalEncoding │      0 │ train │     0 │
│ 2 │ transformer_encoder │ TransformerEncoder │  530 K │ train │     0 │
│ 3 │ classifier          │ Linear             │    774 │ train │     0 │
│ 4 │ criterion           │ CrossEntropyLoss   │      0 │ train │     0 │
└───┴─────────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 535 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 535 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 48                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

d:\AI_WORK\ai_env\lib\site-packages\rich\live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

d:\AI_WORK\ai_env\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:429: Consider setting 
`persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.

d:\AI_WORK\ai_env\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:429: Consider setting 
`persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.

Metric val_loss improved. New best score: 1.730
Epoch 0, global step 89: 'val_loss' reached 1.73043 (best 1.73043), saving model to 'd:\\ActionSnap-\\lightning_logs\\version_9\\checkpoints\\transformer-epoch=00-val_loss=1.7304.ckpt' as top 1
Metric val_loss improved by 0.027 >= min_delta = 0.0. New best score: 1.704
Epoch 1, global step 178: 'val_loss' reached 1.70350 (best 1.70350), saving model to 'd:\\ActionSnap-\\lightning_logs\\version_9\\checkpoints\\transformer-epoch=01-val_loss=1.7035.ckpt' as top 1
Metric val_loss improved by 0.042 >= min_delta = 0.0. New best score: 1.662
Epoch 2, global step 267: 'val_loss' reached 1.66167 (best 1.66167), saving model to 'd:\\ActionSnap-\\lightning_logs\\version_9\\checkpoints\\transformer-epoch=02-val_loss=1.6617.ckpt' as top 1
Metric val_loss improved by 0.122 >= min_delta = 0.0. New best score: 1.540
Epoch 3, global step 356: 'val_loss' reached 1.53996 (best 1.53996), saving model to 'd:\\ActionSnap-\\lightning_logs\\version_9\\checkpoint

In [27]:
try:
    from tensorboard import default  # noqa: F401
    print("TensorBoard is available. Run `%load_ext tensorboard` and `%tensorboard --logdir=lightning_logs` in a notebook cell if you want the dashboard.")
except Exception as exc:
    print(f"TensorBoard dashboard skipped: {exc}")

TensorBoard dashboard skipped: No module named 'pkg_resources'


## 11 · Copy best checkpoint to models/saved_model.ckpt

The Flask app expects the checkpoint at exactly `models/saved_model.ckpt`.

In [28]:
import os, shutil

def get_best_ckpt(lightning_logs_dir="lightning_logs"):
    if "checkpoint_callback" in globals() and checkpoint_callback.best_model_path:
        return checkpoint_callback.best_model_path

    candidates = []
    for root, _, files in os.walk(lightning_logs_dir):
        for file in files:
            if file.endswith(".ckpt"):
                full_path = os.path.join(root, file)
                candidates.append((os.path.getmtime(full_path), full_path))

    if not candidates:
        raise FileNotFoundError("No .ckpt files found under lightning_logs")
    return max(candidates)[1]

best = get_best_ckpt()
print("Best checkpoint:", best)

os.makedirs("models", exist_ok=True)
shutil.copy(best, "models/saved_model.ckpt")
print("Saved -> models/saved_model.ckpt")

Best checkpoint: d:\ActionSnap-\lightning_logs\version_9\checkpoints\transformer-epoch=44-val_loss=0.0242.ckpt
Saved -> models/saved_model.ckpt


## 12 · Quick evaluation

In [29]:
import torch
import numpy as np
from transformer import ActionClassificationTransformer, PoseDataModule, LABELS

loaded = ActionClassificationTransformer.load_from_checkpoint('models/saved_model.ckpt')
loaded.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
loaded.to(device)

dm = PoseDataModule(data_root=DATASET_PATH, batch_size=512)
dm.setup()

all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in dm.val_dataloader():
        preds = loaded(X.to(device)).argmax(dim=1).cpu()
        all_preds.append(preds.numpy())
        all_labels.append(y.numpy())

preds  = np.concatenate(all_preds)
labels = np.concatenate(all_labels)
acc    = (preds == labels).mean()
print(f'Test accuracy: {acc:.4f}')

from sklearn.metrics import classification_report
print(classification_report(labels, preds, target_names=LABELS))

d:\AI_WORK\ai_env\lib\site-packages\lightning_fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Test accuracy: 0.9903
                precision    recall  f1-score   support

       JUMPING       1.00      1.00      1.00       676
 JUMPING_JACKS       1.00      1.00      1.00       783
        BOXING       0.99      0.96      0.98      1216
 WAVING_2HANDS       1.00      1.00      1.00      1320
  WAVING_1HAND       1.00      1.00      1.00      1313
CLAPPING_HANDS       0.91      0.98      0.94       443

      accuracy                           0.99      5751
     macro avg       0.98      0.99      0.99      5751
  weighted avg       0.99      0.99      0.99      5751

